# tally-cam · 처음 보는 대상 세기 — 2단계(글자로 지정) 검출기 비교 (Colab GPU)

대상 클래스로 학습된 검출기를 **쓰지 않고**, 글자로만 대상을 알려 주는 검출기(Grounding DINO tiny, OWLv2 base)로 세어 봅니다.
추적·세기 규칙·채점은 차량과 똑같고, 평가는 각 클립 **뒤 절반**(앞 절반은 점 예산 학습용으로 남겨 둠)입니다.

준비: Drive `tally-cam/samples/`에 차량 dev 영상 5개(미세조정 노트북과 같은 것). 보행자(MOT17)는 노트북이 직접 받습니다(약 6GB, 5분).
런타임: GPU(T4 이상). 전체 약 1시간.

In [ ]:
!nvidia-smi --query-gpu=name --format=csv
from google.colab import drive; drive.mount('/content/drive'); DRIVE='/content/drive/MyDrive/tally-cam'
!git clone -q https://github.com/flytothejy/tally-cam.git /content/tally-cam
%cd /content/tally-cam
!pip install -q transformers ultralytics opencv-python-headless scipy
!mkdir -p samples && cp $DRIVE/samples/*.mp4 samples/

In [ ]:
# 보행자 데이터(MOT17 train) 받기 + 자동 선·정답 만들기
!mkdir -p /content/datasets && cd /content/datasets && wget -q https://motchallenge.net/data/MOT17.zip && unzip -q MOT17.zip "MOT17/train/*FRCNN/*" && rm MOT17.zip
!python bench/mot/prepare_mot.py --root /content/datasets/MOT17/train

In [ ]:
# 검출(뒤 절반만) → 세기 → 뒤 절반 채점
import json, subprocess
PROMPT = {'vehicle': 'car. truck. bus.', 'person': 'person.'}
SETS = [('bench/dataset.json', 'dev', 'vehicle', ['--agnostic', '--ext', '30', '--oneway']),
        ('bench/dataset_mot.json', 'mot', 'person', ['--agnostic', '--anchor', 'bottom'])]
for dsf, split, tgt, rule in SETS:
    for c in json.load(open(dsf))['clips']:
        if c['split'] != split or not c['gt']: continue
        import cv2; cap = cv2.VideoCapture(c['video']); dur = cap.get(7) / cap.get(5)
        for be in ['gdino', 'owlv2']:
            cache = f"bench/cache/{c['id']}_{be}.npz"
            subprocess.run(['python', 'bench/methods/detcache.py', '--video', c['video'], '--backend', be, '--model', 'x', '--prompt', PROMPT[tgt],
                            '--conf', '0.1', '--from-seconds', str(dur / 2), '--out', cache], check=True)
            for conf in ['0.2', '0.3', '0.4']:
                tag = ('BL-MOT-' if tgt == 'person' else 'BL-') + f'{be}-c{conf}'
                subprocess.run(['python', 'bench/methods/track_count.py', '--cache', cache, '--lines', c['gates'], '--out', f"bench/preds/{tag}/{c['id']}.json", '--conf', conf] + rule, check=True)
        print('done', c['id'], flush=True)

In [ ]:
!python bench/analysis/score_half.py --dataset bench/dataset.json --split dev --method BL-gdino-c0.2 --method BL-gdino-c0.3 --method BL-gdino-c0.4 --method BL-owlv2-c0.2 --method BL-owlv2-c0.3 --method BL-owlv2-c0.4 --json /content/bl_vehicle.json
!python bench/analysis/score_half.py --dataset bench/dataset_mot.json --split mot --method BL-MOT-gdino-c0.2 --method BL-MOT-gdino-c0.3 --method BL-MOT-gdino-c0.4 --method BL-MOT-owlv2-c0.2 --method BL-MOT-owlv2-c0.3 --method BL-MOT-owlv2-c0.4 --json /content/bl_mot.json
!mkdir -p $DRIVE/blind && cp /content/bl_vehicle.json /content/bl_mot.json $DRIVE/blind/ && tar czf $DRIVE/blind/preds_blind.tgz bench/preds/BL-*
print('Drive tally-cam/blind 폴더의 bl_vehicle.json, bl_mot.json을 Claude에게 보내 주세요.')